# Agente Funcional — Centro Médico Vitalis

Evaluación Parcial N°2 — **Ingeniería de Soluciones con IA (ISY0101)**
Encargo: *Desarrollo de un Agente Funcional*

Este notebook evoluciona el sistema construido en EP1 (router + RAG) hacia un
**agente funcional con uso autónomo de herramientas**, construido sobre un
framework de agentes real (`langchain.agents` con *tool calling*), en vez de
un flujo if/else manual.

**Qué cambia respecto a EP1:**

| EP1 | EP2 |
|---|---|
| Router manual (clasificador + if/else) | Agente con `AgentExecutor` que decide qué herramienta usar |
| Solo lectura (RAG) | Lectura (RAG, memoria) **y** escritura (registro, redacción de mensajes) |
| Sin memoria entre turnos | Memoria de **corto plazo** (conversación) y **largo plazo** (archivo JSON por paciente) |
| Flujo fijo | Planificación adaptativa: si una herramienta indica que algo no es posible, el agente reformula el plan |

La base de conocimiento (RAG sobre documentos administrativos del centro) y la
conexión al LLM se reutilizan de EP1; lo nuevo es la capa de **agente,
herramientas, memoria y planificación**.

## 1. Instalación de dependencias

In [1]:
!pip install -q langchain langchain-openai langchain-core langchain-classic python-dotenv chromadb sentence-transformers pandas pypdf

## 2. Configuración de credenciales y del modelo LLM

In [2]:
# --- Credenciales: funciona en local (.env) y en Google Colab (Secrets) ---
import os

try:
    from google.colab import userdata          # Colab: panel 🔑 Secrets
    for _k in ("LLM_API_KEY", "LLM_BASE_URL"):
        try:
            os.environ[_k] = userdata.get(_k)
        except Exception:
            pass                                 # el Secret no existe: se usa el default
    os.environ.setdefault("LLM_BASE_URL", "https://openrouter.ai/api/v1")
except ImportError:
    from dotenv import load_dotenv              # Local: archivo .env en la raíz
    load_dotenv()

# Modelo recomendado por el profesor (precio/calidad). No es un dato
# sensible, se fija directo en el código.
os.environ["LLM_MODEL"] = "openai/gpt-6-luna"

from langchain_openai import ChatOpenAI

try:
    from langchain_core.messages import HumanMessage, AIMessage
except ImportError:
    try:
        from langchain.schema import HumanMessage, AIMessage
    except ImportError:
        from langchain_classic.schema import HumanMessage, AIMessage

import json
import re
from datetime import datetime

# 'reasoning' solo lo acepta OpenRouter; otros proveedores responden 400
_es_openrouter = "openrouter.ai" in (os.getenv("LLM_BASE_URL") or "")

llm = ChatOpenAI(
    base_url=os.getenv("LLM_BASE_URL"),
    api_key=os.getenv("LLM_API_KEY"),
    model=os.getenv("LLM_MODEL"),
    temperature=0.2,
    max_retries=3,
    timeout=30,
    extra_body={"reasoning": {"effort": "low"}} if _es_openrouter else None,
)

print("✓ Modelo LLM configurado:", os.getenv("LLM_MODEL"))
print("✓ Base URL:", os.getenv("LLM_BASE_URL"))
print("✓ OpenRouter:", _es_openrouter)
print("✓ Variable 'llm' creada correctamente:", llm is not None)

✓ Modelo LLM configurado: openai/gpt-6-luna
✓ Base URL: https://openrouter.ai/api/v1
✓ OpenRouter: True
✓ Variable 'llm' creada correctamente: True


## 3. Base documental (RAG) — conocimiento administrativo del centro

Se reutiliza la base documental de EP1. Cada fuente es su **propio documento**
independiente (no se concatenan entre sí): mezclar un resumen corto con PDFs
largos en el mismo documento diluye la recuperación del chunk relevante
cuando se indexan juntos (problema detectado y corregido en EP1).

Si tienes los PDFs reales de EP1 (`Ayuno.pdf`, `i_009_sedimento_de_orina.pdf`,
`Bono_de_Fonasa.pdf`, `Comprar_un_bono_de_Fonasa.pdf`), puedes subirlos con la
celda de `files.upload()` de abajo. Si no los tienes a mano, el notebook sigue
funcionando igual con los documentos simulados (horarios, convenios, FAQ).

In [3]:
import re as _re

def buscar_archivo(nombre_base, archivos_subidos):
    """Encuentra un archivo aunque Colab le haya agregado un sufijo (N) por
    haberse subido más de una vez sin reiniciar el entorno de ejecución."""
    for nombre in archivos_subidos:
        limpio = _re.sub(r"\s*\(\d+\)(?=\.\w+$)", "", nombre)
        if limpio.lower() == nombre_base.lower():
            return nombre
    return None


def extraer_texto_pdf(ruta):
    from pypdf import PdfReader
    lector = PdfReader(ruta)
    return "\n".join((pagina.extract_text() or "") for pagina in lector.pages)


archivos_subidos = []
try:
    from google.colab import files
    print("Sube los PDFs reales de EP1 si los tienes (opcional, puedes cancelar):")
    subidos = files.upload()
    archivos_subidos = list(subidos.keys())
except Exception:
    print("Entorno sin subida interactiva de archivos (no Colab, o se omitió). "
          "Se usará solo la base documental simulada.")

DOCUMENTOS = [
    {
        "id": "horarios_vitalis",
        "fuente": "Horarios_Vitalis.pdf",
        "contenido": """
Horarios de atención - Centro Médico Vitalis

Dr. Andrés Soto (Medicina General): Lunes a viernes, 09:00 a 13:00 y 15:00 a 18:00 hrs.
Nutricionista Camila Ríos: Martes y jueves, 10:00 a 17:00 hrs.
Kinesiólogo Felipe Muñoz: Lunes, miércoles y viernes, 08:30 a 16:00 hrs.

El centro no atiende urgencias. Para emergencias médicas dirigirse al servicio
de urgencia más cercano o llamar al 131 (SAMU).
""",
    },
    {
        "id": "convenios_vitalis",
        "fuente": "Convenios_Vitalis.pdf",
        "contenido": """
Convenios y valores - Centro Médico Vitalis

Se atiende Fonasa (bonificación según tramo), convenios con Isapre Colmena,
Isapre Cruz Blanca e Isapre Consalud, y modalidad particular.

Valores referenciales particulares:
- Consulta Medicina General: $25.000
- Consulta Nutrición: $22.000
- Sesión de Kinesiología: $18.000

Los convenios con isapres pueden variar el copago según el plan de cada paciente;
se recomienda confirmar bonificación exacta al momento de agendar.
""",
    },
    {
        "id": "faq_vitalis",
        "fuente": "FAQ_Recepcion_Vitalis.csv",
        "contenido": """
Preguntas frecuentes recopiladas por recepción:

¿Debo traer algo a la primera consulta? Sí, cédula de identidad y carné de Fonasa
o Isapre si corresponde. Si trae exámenes previos, puede llevarlos.

¿Atienden los fines de semana? No, el centro atiende de lunes a viernes según
horario de cada profesional.

¿Puedo cambiar mi hora agendada? Sí, contactando a recepción con al menos
24 horas de anticipación.
""",
    },
]

# PDFs reales (EP1), cada uno como documento independiente, solo si se subieron
_pdfs_opcionales = [
    ("Ayuno.pdf", "guia_ayuno_falp", "Instructivo_Ayuno_FALP.pdf"),
    ("i_009_sedimento_de_orina.pdf", "guia_orina_bupa", "Instructivo_Orina_Bupa.pdf"),
    ("Bono_de_Fonasa.pdf", "bono_fonasa_chileatiende", "Bono_Fonasa_ChileAtiende.pdf"),
    ("Comprar_un_bono_de_Fonasa.pdf", "bono_fonasa_previsionsocial", "Comprar_Bono_Fonasa_PrevisionSocial.pdf"),
]
for nombre_base, doc_id, fuente_label in _pdfs_opcionales:
    encontrado = buscar_archivo(nombre_base, archivos_subidos)
    if encontrado:
        DOCUMENTOS.append({
            "id": doc_id,
            "fuente": fuente_label,
            "contenido": extraer_texto_pdf(encontrado),
        })

print(f"Documentos cargados: {len(DOCUMENTOS)}")
for d in DOCUMENTOS:
    print(" -", d["id"], "|", d["fuente"])

Sube los PDFs reales de EP1 si los tienes (opcional, puedes cancelar):


Saving Ayuno.pdf to Ayuno (6).pdf
Saving Bono de Fonasa.pdf to Bono de Fonasa (6).pdf
Saving Comprar un bono de Fonasa.pdf to Comprar un bono de Fonasa (6).pdf
Saving i_009_sedimento_de_orina.pdf to i_009_sedimento_de_orina (6).pdf
Documentos cargados: 5
 - horarios_vitalis | Horarios_Vitalis.pdf
 - convenios_vitalis | Convenios_Vitalis.pdf
 - faq_vitalis | FAQ_Recepcion_Vitalis.csv
 - guia_ayuno_falp | Instructivo_Ayuno_FALP.pdf
 - guia_orina_bupa | Instructivo_Orina_Bupa.pdf


## 4. Chunking e indexación vectorial (ChromaDB, embeddings multilingües)

In [4]:
def chunk_text(texto, chunk_size=400, overlap=50):
    """Divide un texto en fragmentos con solape, usando longitud de caracteres
    por simplicidad (en producción conviene tokenizar según el modelo de
    embeddings usado)."""
    texto = texto.strip()
    chunks, start = [], 0
    while start < len(texto):
        end = start + chunk_size
        chunks.append(texto[start:end].strip())
        start += chunk_size - overlap
    return [c for c in chunks if c]


chunks_indexados = []
for doc in DOCUMENTOS:
    for i, chunk in enumerate(chunk_text(doc["contenido"])):
        chunks_indexados.append({
            "chunk_id": f"{doc['id']}_{i}",
            "fuente": doc["fuente"],
            "texto": chunk,
        })

import chromadb
from chromadb.utils import embedding_functions

# Modelo multilingüe: el modelo en inglés (all-MiniLM-L6-v2) degradaba la
# calidad de recuperación para consultas en español (detectado en EP1).
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

chroma_client = chromadb.Client()  # en memoria; usar PersistentClient para persistir en disco
coleccion = chroma_client.get_or_create_collection(
    name="vitalis_docs",
    embedding_function=embedding_fn,
)
coleccion.add(
    ids=[c["chunk_id"] for c in chunks_indexados],
    documents=[c["texto"] for c in chunks_indexados],
    metadatas=[{"fuente": c["fuente"]} for c in chunks_indexados],
)

print(f"Colección indexada con {coleccion.count()} fragmentos de {len(DOCUMENTOS)} documentos.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Colección indexada con 19 fragmentos de 5 documentos.


## 5. Memoria del agente — corto y largo plazo

- **Memoria de corto plazo**: historial de la conversación actual (lista de
  mensajes), se pierde al terminar la sesión. Le da contexto al agente dentro
  de un mismo intercambio con el paciente.
- **Memoria de largo plazo**: un archivo **JSON** (`memoria_largo_plazo_vitalis.json`)
  indexado por **RUT del paciente**, que persiste entre sesiones (se guarda a
  disco; en Colab hay que descargarlo/volver a subirlo entre sesiones, o
  montar Drive para que quede fijo). Guarda antecedentes relevantes ya
  conversados con cada paciente, para no volver a preguntarlos.

Se eligió un archivo simple en vez de una colección vectorial separada porque
el volumen de datos por paciente es bajo y la recuperación es por clave exacta
(el RUT), no semántica. El RUT se usa como identificador porque es el dato
real con el que un centro médico chileno identifica a sus pacientes.

In [5]:
import os
import sqlite3
import uuid

# ---------------------------------------------------------------------------
# Carpeta de persistencia: Drive en Colab, carpeta actual en local
# ---------------------------------------------------------------------------
try:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA = "/content/drive/MyDrive/vitalis"
    os.makedirs(CARPETA, exist_ok=True)
except ImportError:
    CARPETA = "."


def normalizar_rut(rut):
    """Normaliza un RUT a mayúsculas, sin puntos ni espacios (ej. '12.345.678-9'
    y '12345678-9' deben apuntar al mismo registro en la memoria de largo
    plazo, para no duplicarlo por una simple diferencia de formato)."""
    return rut.replace(".", "").replace(" ", "").upper()


RUTA_MEMORIA_LARGO_PLAZO = os.path.join(CARPETA, "memoria_largo_plazo_vitalis.json")

def cargar_memoria_largo_plazo():
    if os.path.exists(RUTA_MEMORIA_LARGO_PLAZO):
        with open(RUTA_MEMORIA_LARGO_PLAZO, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}


def guardar_memoria_largo_plazo(memoria):
    with open(RUTA_MEMORIA_LARGO_PLAZO, "w", encoding="utf-8") as f:
        json.dump(memoria, f, ensure_ascii=False, indent=2)


def actualizar_memoria_paciente(rut, nota):
    rut = normalizar_rut(rut)
    memoria = cargar_memoria_largo_plazo()
    registro = memoria.setdefault(rut, {"notas": []})
    registro["notas"].append({
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "nota": nota,
    })
    guardar_memoria_largo_plazo(memoria)


# ---------------------------------------------------------------------------
# Persistencia del historial de conversación en SQLite
# ---------------------------------------------------------------------------
DB_PATH = os.path.join(CARPETA, "memoria_vitalis.db")
SESION_ID = uuid.uuid4().hex[:8]   # identifica esta ejecución del notebook

def _db():
    con = sqlite3.connect(DB_PATH)
    con.execute("""CREATE TABLE IF NOT EXISTS mensajes (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        rut TEXT NOT NULL,
        sesion TEXT NOT NULL,
        rol TEXT NOT NULL,
        contenido TEXT NOT NULL,
        fecha TEXT DEFAULT CURRENT_TIMESTAMP)""")
    return con


def guardar_turno_db(rut, consulta, respuesta):
    con = _db()
    con.executemany(
        "INSERT INTO mensajes (rut, sesion, rol, contenido) VALUES (?,?,?,?)",
        [(rut, SESION_ID, "humano", consulta), (rut, SESION_ID, "agente", respuesta)],
    )
    con.commit()
    con.close()


def cargar_historial_db(rut):
    # Las conversaciones anónimas no se recargan: serían de otra persona.
    if rut == "anonimo":
        return []
    con = _db()
    filas = con.execute(
        "SELECT rol, contenido FROM mensajes WHERE rut=? ORDER BY id", (rut,)
    ).fetchall()
    con.close()
    return filas


def migrar_historial_db(rut_destino):
    """Pasa al RUT los turnos anónimos de ESTA sesión (no los de otras)."""
    con = _db()
    con.execute(
        "UPDATE mensajes SET rut=? WHERE rut='anonimo' AND sesion=?",
        (rut_destino, SESION_ID),
    )
    con.commit()
    con.close()


# Memoria de corto plazo: una ConversationSummaryBufferMemory por paciente
# (clave: RUT). Vive en RAM, pero se reconstruye desde SQLite al iniciar,
# así el historial sobrevive a un reinicio del notebook.
try:
    from langchain.memory import ConversationSummaryBufferMemory
except ImportError:
    from langchain_classic.memory import ConversationSummaryBufferMemory

from langchain_openai import ChatOpenAI as _ChatOpenAI

class ChatOpenAIConConteoAprox(_ChatOpenAI):
    """ChatOpenAI no sabe contar tokens para modelos con nombre no-OpenAI
    (ej. 'openai/gpt-6-luna' en OpenRouter) y lanza NotImplementedError, que
    es justo lo que ConversationSummaryBufferMemory necesita para decidir
    cuándo resumir. Se usa una aproximación (~4 caracteres por token)."""
    def get_num_tokens(self, text):
        return max(1, len(text) // 4)

    def get_num_tokens_from_messages(self, messages, *args, **kwargs):
        return sum(self.get_num_tokens(str(m.content)) + 3 for m in messages)


llm_memoria = ChatOpenAIConConteoAprox(
    base_url=os.getenv("LLM_BASE_URL"),
    api_key=os.getenv("LLM_API_KEY"),
    model=os.getenv("LLM_MODEL"),
    temperature=0.2,
    max_retries=3,
    timeout=30,
    extra_body=getattr(llm, "extra_body", None),
)

memoria_corto_plazo = {}  # {rut: ConversationSummaryBufferMemory}

def obtener_memoria_corto_plazo(rut):
    rut = normalizar_rut(rut)
    if rut not in memoria_corto_plazo:
        memoria = ConversationSummaryBufferMemory(
            llm=llm_memoria,
            max_token_limit=800,  # sobre este límite, los turnos antiguos se resumen
            return_messages=True,
            memory_key="chat_history",
        )
        # Reconstruye el historial guardado en SQLite (si lo hay)
        for rol, contenido in cargar_historial_db(rut):
            if rol == "humano":
                memoria.chat_memory.add_user_message(contenido)
            else:
                memoria.chat_memory.add_ai_message(contenido)
        try:
            memoria.prune()   # resume lo antiguo si lo recargado supera el límite
        except Exception:
            pass
        memoria_corto_plazo[rut] = memoria
    return memoria_corto_plazo[rut]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 6. Herramientas (tools) del agente

Cinco herramientas, cubriendo los tres tipos pedidos por la pauta:

| Herramienta | Tipo | Qué hace |
|---|---|---|
| `buscar_informacion_clinica` | Consulta | RAG sobre documentos administrativos (reutiliza la colección de ChromaDB) |
| `consultar_memoria_paciente` | Consulta | Recupera antecedentes guardados del paciente en la memoria de largo plazo |
| `verificar_disponibilidad` | Razonamiento | Revisa disponibilidad simulada de horas; habilita re-planificación si no hay cupo |
| `registrar_solicitud` | Escritura | Deja registrada una solicitud de hora/derivación para recepción |
| `redactar_mensaje_recepcion` | Escritura | Redacta un mensaje para que recepción lo revise y envíe |

In [6]:
from langchain_core.tools import tool


@tool
def buscar_informacion_clinica(consulta: str) -> str:
    """Busca información administrativa del Centro Médico Vitalis (horarios,
    convenios, valores, preparación de exámenes) en la base documental
    indexada. Úsala para cualquier pregunta sobre el funcionamiento del
    centro. NO la uses para responder preguntas clínicas (síntomas,
    diagnósticos, tratamientos): esas se derivan, nunca se contestan con
    esta herramienta."""
    resultados = coleccion.query(query_texts=[consulta], n_results=4)
    fragmentos = resultados["documents"][0]
    fuentes = [m["fuente"] for m in resultados["metadatas"][0]]
    bloques = [f"[Documento: {f}] {frag}" for frag, f in zip(fragmentos, fuentes)]
    return "\n---\n".join(bloques) if bloques else "Sin resultados en la base documental."


@tool
def consultar_memoria_paciente(rut: str) -> str:
    """Recupera antecedentes previamente guardados de un paciente específico
    (identificado por su RUT, ej. '12.345.678-9') desde la memoria de largo
    plazo. Úsala al inicio de una conversación con un paciente que entregó su
    RUT, para no volver a preguntar algo que ya se registró en una sesión
    anterior."""
    rut = normalizar_rut(rut)
    memoria = cargar_memoria_largo_plazo()
    registro = memoria.get(rut)
    if not registro or not registro.get("notas"):
        return f"No hay antecedentes guardados para el RUT {rut}."
    return "\n".join(f"- {n['nota']}" for n in registro["notas"])


# Disponibilidad simulada: deliberadamente SIN cupo para un caso específico,
# para poder demostrar la re-planificación adaptativa del agente.
_DISPONIBILIDAD_SIMULADA = {
    ("Dr. Andrés Soto", "lunes"): {"disponible": False, "alternativas": ["miércoles 10:00", "viernes 16:00"]},
    ("Dr. Andrés Soto", "miércoles"): {"disponible": True, "alternativas": []},
    ("Nutricionista Camila Ríos", "martes"): {"disponible": True, "alternativas": []},
    ("Kinesiólogo Felipe Muñoz", "lunes"): {"disponible": True, "alternativas": []},
}

@tool
def verificar_disponibilidad(profesional: str, dia: str) -> str:
    """Verifica si hay horas disponibles para un profesional en un día dado
    (ej. profesional='Dr. Andrés Soto', dia='lunes'). Úsala SIEMPRE antes de
    registrar una solicitud de hora, para saber si hay que ofrecer una
    alternativa en vez de confirmar directamente."""
    clave = (profesional.strip(), dia.strip().lower())
    info = _DISPONIBILIDAD_SIMULADA.get(clave)
    if info is None:
        return (f"Sin datos de disponibilidad exacta para {profesional} el {dia}. "
                f"Sugerir confirmar directamente con recepción.")
    if info["disponible"]:
        return f"Hay disponibilidad para {profesional} el {dia}."
    alternativas = ", ".join(info["alternativas"]) or "sin alternativas registradas"
    return (f"NO hay disponibilidad para {profesional} el {dia}. "
            f"Alternativas sugeridas: {alternativas}.")


solicitudes_registradas = []

@tool
def registrar_solicitud(rut: str, tipo: str, detalle: str) -> str:
    """Registra formalmente una solicitud (de hora, de derivación, o de
    información adicional) para que recepción la procese. rut es el RUT del
    paciente (ej. '12.345.678-9'). tipo debe ser uno de: 'hora', 'derivacion',
    'otro'. Úsala una vez que ya se sabe qué se le va a pedir a recepción
    (por ejemplo, después de verificar disponibilidad)."""
    rut = normalizar_rut(rut)
    solicitudes_registradas.append({
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "rut": rut,
        "tipo": tipo,
        "detalle": detalle,
    })
    return f"Solicitud registrada para el RUT {rut} (tipo: {tipo})."


mensajes_borrador = []

@tool
def redactar_mensaje_recepcion(rut: str, motivo: str) -> str:
    """Redacta un mensaje breve y cordial dirigido a recepción, explicando lo
    que el paciente (identificado por su RUT) necesita, para que una persona
    lo revise y lo envíe. No envía el mensaje: solo lo deja redactado como
    borrador."""
    rut = normalizar_rut(rut)
    prompt = (
        "Redacta un mensaje breve (máximo 3 líneas), cordial y profesional, "
        "dirigido al equipo de recepción del Centro Médico Vitalis, "
        f"explicando que el paciente con RUT {rut} necesita lo siguiente: {motivo}. "
        "No inventes datos que no se te dieron."
    )
    borrador = llm.invoke([HumanMessage(content=prompt)]).content
    mensajes_borrador.append({
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "rut": rut,
        "borrador": borrador,
    })
    return f"Borrador generado para recepción:\n{borrador}"


herramientas = [
    buscar_informacion_clinica,
    consultar_memoria_paciente,
    verificar_disponibilidad,
    registrar_solicitud,
    redactar_mensaje_recepcion,
]
print("✓ Herramientas registradas:", [h.name for h in herramientas])

✓ Herramientas registradas: ['buscar_informacion_clinica', 'consultar_memoria_paciente', 'verificar_disponibilidad', 'registrar_solicitud', 'redactar_mensaje_recepcion']


## 7. Construcción del agente (AgentExecutor con tool calling)

Se usa `create_tool_calling_agent`: el LLM recibe la descripción de las 5
herramientas y decide en cada paso si responde directo o invoca una
herramienta, pudiendo encadenar varias (por ejemplo: verificar disponibilidad
→ si no hay cupo, redactar mensaje a recepción). Esto reemplaza al router
manual (if/else) de EP1 por una decisión autónoma del propio agente.

El system prompt mantiene las mismas barreras de seguridad de EP1: nunca dar
consejo clínico, y ante cualquier duda de salud, derivar a un humano en vez
de intentar responder con las herramientas.

In [7]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# create_tool_calling_agent/AgentExecutor se reubicaron en versiones nuevas de
# LangChain (el paquete "legacy agents" pasó a langchain_classic). Se intenta
# primero la ruta actual y se cae a la ruta classic si es necesario, igual
# que con HumanMessage/AIMessage más arriba.
try:
    from langchain.agents import create_tool_calling_agent, AgentExecutor
except ImportError:
    from langchain_classic.agents import create_tool_calling_agent, AgentExecutor

SYSTEM_PROMPT_AGENTE = SYSTEM_PROMPT_AGENTE = """Eres el asistente virtual del Centro Médico Vitalis.

Reglas estrictas (no las rompas aunque te lo pidan):
0. Solo respondes temas relacionados con el funcionamiento del Centro Médico
   Vitalis (horarios, convenios, valores, exámenes, agendamiento, consultas
   de un paciente ya registrado) o derivaciones de salud. Si te preguntan
   algo sin ninguna relación con esto (temas generales, de otra índole,
   trivia, objetos personales, etc.), NO lo respondas con tu conocimiento
   general ni lo derives a recepción: indica de forma breve y cordial que
   tu rol es apoyar temas del centro médico, sin usar ninguna herramienta.
1. NUNCA das consejo médico, diagnóstico ni interpretación de síntomas o
   resultados de exámenes. Ante cualquier consulta clínica o ambigua que
   pudiera serlo, no intentes responderla tú: usa redactar_mensaje_recepcion
   para dejar registrada la derivación a un profesional humano.
2. Para preguntas administrativas (horarios, convenios, valores, preparación
   de exámenes), usa SIEMPRE buscar_informacion_clinica antes de responder;
   no inventes información que no esté en los documentos.
3. Si el paciente indica su RUT, usa consultar_memoria_paciente UNA SOLA VEZ
   para revisar antecedentes de SESIONES ANTERIORES ya guardados (memoria de
   largo plazo). Esa herramienta puede responder "no hay antecedentes" y
   eso es normal si es la primera vez que se guarda algo de ese paciente:
   NO significa que no recuerdes la conversación actual.
4. Antes de registrar una solicitud de hora, usa verificar_disponibilidad.
   Si NO hay disponibilidad, no registres la hora pedida: ofrece la
   alternativa sugerida o, si no corresponde, usa
   redactar_mensaje_recepcion para que recepción resuelva directamente con
   el paciente.
5. Sé breve, claro y cordial. Cita la fuente del documento cuando uses
   buscar_informacion_clinica.
6. Para recordar lo que se ha conversado en ESTA MISMA sesión (incluso antes
   de que el paciente diera su RUT), usa directamente el historial de la
   conversación que ya tienes disponible: no necesitas ninguna herramienta
   para eso, y nunca digas que "no recuerdas" algo que ya está en el
   historial de esta charla.
"""

prompt_agente = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT_AGENTE),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, herramientas, prompt_agente)
agent_executor = AgentExecutor(
    agent=agent,
    tools=herramientas,
    verbose=False,
    return_intermediate_steps=True,  # necesario para la trazabilidad (sección 9)
    max_iterations=6,
)

print("✓ AgentExecutor construido con", len(herramientas), "herramientas.")

✓ AgentExecutor construido con 5 herramientas.


## 8. Trazabilidad extendida (pasos del agente, herramientas invocadas)

In [8]:
registro_log_agente = []

def registrar_interaccion_agente(rut, consulta, resultado):
    pasos = resultado.get("intermediate_steps", [])
    herramientas_usadas = [accion.tool for accion, _obs in pasos]
    registro_log_agente.append({
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "rut": normalizar_rut(rut),
        "consulta": consulta,
        "herramientas_usadas": ", ".join(herramientas_usadas) if herramientas_usadas else "(ninguna)",
        "n_pasos": len(pasos),
        "respuesta": resultado["output"],
    })

## 9. Orquestador: ejecución con memoria de corto y largo plazo

In [9]:
_PATRON_RUT = re.compile(r"\b(\d{1,2}\.?\d{3}\.?\d{3})-?([\dkK])\b")

def extraer_rut_de_texto(texto):
    """Busca un RUT dentro de un texto libre, para poder vincular la
    conversación al RUT correcto aunque no se haya entregado al principio."""
    match = _PATRON_RUT.search(texto)
    if not match:
        return None
    cuerpo, verificador = match.groups()
    return normalizar_rut(f"{cuerpo}-{verificador}")


def agente_vitalis_v2(consulta, rut="anonimo"):
    rut = normalizar_rut(rut)

    # Si la conversación empezó sin RUT (o con uno distinto) y el paciente
    # recién ahora lo menciona, se migran los turnos de esta sesión a ese RUT
    # (en SQLite) y se reconstruye su memoria: queda su historial anterior
    # (si lo tenía) más lo conversado hasta ahora.
    rut_detectado = extraer_rut_de_texto(consulta)
    if rut_detectado and rut_detectado != rut:
        if rut == "anonimo":
            migrar_historial_db(rut_detectado)
        memoria_corto_plazo.pop(rut, None)
        memoria_corto_plazo.pop(rut_detectado, None)
        rut = rut_detectado

    memoria = obtener_memoria_corto_plazo(rut)
    historial = memoria.load_memory_variables({})["chat_history"]

    resultado = agent_executor.invoke({
        "input": consulta,
        "chat_history": historial,
    })

    # Actualiza la memoria de corto plazo (resume si supera el límite de
    # tokens) y guarda el turno en SQLite.
    memoria.save_context({"input": consulta}, {"output": resultado["output"]})
    guardar_turno_db(rut, consulta, resultado["output"])

    # Trazabilidad
    registrar_interaccion_agente(rut, consulta, resultado)

    resultado["rut_efectivo"] = rut  # informa al llamador si el RUT cambió
    return resultado

## 10. Escenario de prueba: planificación adaptativa

Este caso demuestra que el agente **re-planifica** cuando una herramienta le
informa que el plan inicial no es viable: el paciente pide hora con el
Dr. Soto el día lunes (deliberadamente marcado como sin cupo en la
disponibilidad simulada). Se espera que el agente:

1. Llame `verificar_disponibilidad` → recibe que no hay cupo y sí alternativas.
2. En vez de fallar o registrar igual la hora pedida, **reformule el plan**:
   ofrezca la alternativa al paciente, o bien use `redactar_mensaje_recepcion`
   si la situación lo amerita.

In [10]:
print("=== Escenario: planificación adaptativa (sin cupo disponible) ===\n")
resultado_demo = agente_vitalis_v2(
    "Hola, mi RUT es 9.876.543-2. Quiero agendar hora con el Dr. Andrés Soto el día lunes.",
    rut="9.876.543-2",
)
print("Respuesta del agente:\n", resultado_demo["output"])
print("\nPasos ejecutados (herramienta → resultado):")
for accion, observacion in resultado_demo["intermediate_steps"]:
    print(f" - {accion.tool}({accion.tool_input}) -> {observacion}")

=== Escenario: planificación adaptativa (sin cupo disponible) ===



/tmp/ipykernel_16465/784989448.py:137: LangChainDeprecationWarning: The class `ConversationSummaryBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memoria = ConversationSummaryBufferMemory(


Respuesta del agente:
 No hay horas disponibles con el Dr. Andrés Soto el lunes. Como alternativa, hay disponibilidad el miércoles a las 10:00 o el viernes a las 16:00. ¿Cuál prefieres?

Pasos ejecutados (herramienta → resultado):
 - consultar_memoria_paciente({'rut': '9.876.543-2'}) -> No hay antecedentes guardados para el RUT 9876543-2.
 - verificar_disponibilidad({'profesional': 'Dr. Andrés Soto', 'dia': 'lunes'}) -> NO hay disponibilidad para Dr. Andrés Soto el lunes. Alternativas sugeridas: miércoles 10:00, viernes 16:00.


## 11. Casos de prueba generales (cobertura de las 5 herramientas)

In [11]:
casos_de_prueba = [
    ("¿Qué horario atiende la nutricionista Camila Ríos?", "15.111.222-3"),
    ("Mi RUT es 12.345.678-9, ¿tengo algún antecedente registrado de una visita anterior?", "12.345.678-9"),
    ("Me duele mucho la cabeza hace tres días y tengo fiebre, ¿qué puede ser?", "16.444.555-6"),
    ("¿Atienden Isapre Colmena?", "17.777.888-k"),
    ("Necesito hora con el Kinesiólogo Felipe Muñoz el día lunes.", "18.999.000-1"),
]

for consulta, rut in casos_de_prueba:
    print(f"--- {rut}: {consulta}")
    r = agente_vitalis_v2(consulta, rut=rut)
    print(r["output"])
    print()

--- 15.111.222-3: ¿Qué horario atiende la nutricionista Camila Ríos?
La nutricionista Camila Ríos atiende los martes y jueves, de 10:00 a 17:00 hrs.  
Fuente: *Horarios_Vitalis.pdf*.

--- 12.345.678-9: Mi RUT es 12.345.678-9, ¿tengo algún antecedente registrado de una visita anterior?
No hay antecedentes de visitas anteriores guardados para tu RUT.

--- 16.444.555-6: Me duele mucho la cabeza hace tres días y tengo fiebre, ¿qué puede ser?
No puedo determinar la causa de esos síntomas. Dejé un borrador para recepción solicitando que un profesional evalúe tu caso. Si el dolor es muy intenso o empeora, o tienes confusión, rigidez de cuello, vómitos persistentes o dificultad para respirar, busca atención de urgencia ahora.

--- 17.777.888-k: ¿Atienden Isapre Colmena?
Sí, el Centro Médico Vitalis tiene convenio con Isapre Colmena. El copago puede variar según tu plan; se recomienda confirmar la bonificación al agendar.  
**Fuente:** *Convenios_Vitalis.pdf*.

--- 18.999.000-1: Necesito hora c

## 12. Exportar registro de trazabilidad

In [16]:
import pandas as pd
import os

def exportar_acumulando(registros, nombre_archivo):
    """Suma los registros de esta sesión a lo guardado en sesiones anteriores
    y sobrescribe el CSV con el total, sin duplicar filas."""
    ruta = os.path.join(CARPETA, nombre_archivo)
    df_nuevo = pd.DataFrame(registros)

    df_previo = pd.DataFrame()
    if os.path.exists(ruta):
        try:
            df_previo = pd.read_csv(ruta)
        except pd.errors.EmptyDataError:
            pass  # archivo vacío de una sesión sin registros: se ignora

    partes = [d for d in (df_previo, df_nuevo) if not d.empty]
    if not partes:
        return pd.DataFrame()   # nada que guardar, no se crea un CSV vacío

    df = pd.concat(partes, ignore_index=True).drop_duplicates().reset_index(drop=True)
    df.to_csv(ruta, index=False)
    return df

df_trazabilidad = exportar_acumulando(
    registro_log_agente, "registro_trazabilidad_agente_vitalis.csv")
df_solicitudes = exportar_acumulando(
    solicitudes_registradas, "registro_solicitudes_vitalis.csv")
df_mensajes = exportar_acumulando(
    mensajes_borrador, "registro_mensajes_borrador_vitalis.csv")

print("Trazabilidad del agente (acumulada):")
print(df_trazabilidad)
print("\nSolicitudes registradas (herramienta de escritura 1):")
print(df_solicitudes)
print("\nMensajes borrador para recepción (herramienta de escritura 2):")
print(df_mensajes)
print("\n✓ CSV guardados en:", os.path.abspath(CARPETA))

Trazabilidad del agente (acumulada):
              timestamp         rut  \
0   2026-10-21T18:47:42   9876543-2   
1   2026-10-21T18:47:46  15111222-3   
2   2026-10-21T18:47:49  12345678-9   
3   2026-10-21T18:47:56  16444555-6   
4   2026-10-21T18:48:00  17777888-K   
5   2026-10-21T18:48:08  18999000-1   
6   2026-10-21T18:48:30  20064625-8   
7   2026-10-21T18:51:53   9876543-2   
8   2026-10-21T18:51:57  15111222-3   
9   2026-10-21T18:52:02  12345678-9   
10  2026-10-21T18:52:12  16444555-6   
11  2026-10-21T18:52:17  17777888-K   
12  2026-10-21T18:52:21  18999000-1   
13  2026-10-21T18:53:17     ANONIMO   

                                             consulta  \
0   Hola, mi RUT es 9.876.543-2. Quiero agendar ho...   
1   ¿Qué horario atiende la nutricionista Camila R...   
2   Mi RUT es 12.345.678-9, ¿tengo algún anteceden...   
3   Me duele mucho la cabeza hace tres días y teng...   
4                           ¿Atienden Isapre Colmena?   
5   Necesito hora con el Kinesiólog

## 13. Haz tus propias preguntas al agente

Ejecuta esta celda para conversar directamente con el agente. Te pedirá un
**RUT** (puedes inventar uno, ej. `15.111.222-3`, o reutilizar
`12.345.678-9` para ver la memoria de largo plazo ya cargada) y luego tus
consultas una por una. Escribe `salir` para terminar.

In [15]:
_entrada_rut_inicial = input("Ingresa tu RUT (o Enter para 'anonimo'): ").strip()
rut_interactivo = extraer_rut_de_texto(_entrada_rut_inicial) or normalizar_rut(_entrada_rut_inicial or "anonimo")

while True:
    consulta_personal = input("Escribe tu consulta ('salir' para terminar): ").strip()
    if consulta_personal.lower() in ("salir", "exit", "quit", ""):
        print("Fin de la conversación interactiva.")
        break

    resultado_interactivo = agente_vitalis_v2(consulta_personal, rut=rut_interactivo)

    if resultado_interactivo["rut_efectivo"] != rut_interactivo:
        rut_interactivo = resultado_interactivo["rut_efectivo"]
        print(f"(RUT detectado: se vincula la conversación a {rut_interactivo})")

    print("\nAgente:", resultado_interactivo["output"])
    if resultado_interactivo["intermediate_steps"]:
        herramientas_usadas = [accion.tool for accion, _obs in resultado_interactivo["intermediate_steps"]]
        print("(herramientas usadas:", ", ".join(herramientas_usadas), ")")
    print()

Ingresa tu RUT (o Enter para 'anonimo'): 
Escribe tu consulta ('salir' para terminar): donde compro zapatos?

Agente: Mi función es apoyar con consultas sobre el Centro Médico Vitalis.

Escribe tu consulta ('salir' para terminar): salir
Fin de la conversación interactiva.


## Notas finales y próximos pasos

- **Framework de agente**: `langchain.agents.create_tool_calling_agent` +
  `AgentExecutor`. El modelo (`openai/gpt-oss-20b` vía Groq) decide de forma
  autónoma qué herramienta(s) invocar y en qué orden, en vez de seguir un
  flujo if/else fijo como en EP1.
- **Memoria**: corto plazo = historial de mensajes de la sesión actual;
  largo plazo = archivo JSON indexado por **RUT del paciente** (el
  identificador real que usa un centro médico chileno), persistente entre
  ejecuciones del notebook (no entre sesiones de Colab distintas, salvo que
  se descargue/suba el archivo o se use Drive).
- **Planificación adaptativa**: demostrada en la sección 10 — el agente
  cambia de plan cuando `verificar_disponibilidad` le indica que la opción
  pedida no existe.
- **Límite conocido**: la memoria de largo plazo es por archivo plano; para
  un volumen real de pacientes convendría una base de datos (SQLite o
  similar) con índices, no solo JSON.
- **Pendiente para la presentación (IE7-IE10)**: demo en vivo de este
  notebook, diagrama de orquestación del agente (herramientas + memoria), y
  justificación técnica de por qué se escogió `tool calling` por sobre un
  agente puramente basado en prompting ReAct manual.